In [8]:

# necessary imports
import os
from dotenv import load_dotenv
from groq import Groq
from sarvamai import SarvamAI
from sarvamai.play import play, save
from tqdm import tqdm

# load environment variables from .env file
load_dotenv()

# retrieve the Groq API key from environment variables
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

# initialize the client with the API key
groq_client = Groq(api_key=GROQ_API_KEY)
sarvamai_client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

In [6]:
# Constants
SYSTEM_PROMPT = """
You are a creative story generator. 
Your task is to write a completely original and random short story of around 400 words each time you are asked. 

Guidelines:
- Each story must have a clear beginning, middle, and end.
- The genre, setting, and characters should be randomly chosen. They can range from fantasy, sci-fi, mystery, romance, adventure, or slice of life — vary them every time.
- The story should read naturally and cohesively, even though it’s random.
- Avoid repeating the same themes or names across different generations.
- Keep the language engaging and descriptive, showing rather than telling.
- Do not include any explanations, introductions, or meta text — only output the story itself.
- The story should be approximately 400 words (±20 words).
- Produce the story in a single line.
"""
USER_PROMPT = """Generate a random short story of about 400 words."""
NUM_STORIES = 100
STORIES_LIST = []
MODEL_NAME = "meta-llama/llama-4-maverick-17b-128e-instruct"

In [ ]:
# generating stories
for _ in tqdm(range(NUM_STORIES), desc="Generating Stories", ncols=100): 
  completion = groq_client.chat.completions.create(
      model=MODEL_NAME,
      messages=[
        {
          "role": "system",
          "content": SYSTEM_PROMPT
        },
        {
          "role": "user",
          "content": USER_PROMPT
        }
      ],
      temperature=1,
      max_completion_tokens=4096,
      top_p=1,
      stream=True,
      stop=None
  )
  full_response = ""
  for chunk in completion:
    if chunk.choices[0].delta.content:
        full_response += chunk.choices[0].delta.content

  STORIES_LIST.append(full_response)

Generating Stories: 100%|█████████████████████████████████████████| 100/100 [05:29<00:00,  3.30s/it]


In [ ]:
count = 0
os.makedirs("audio_samples", exist_ok=True)
for text in tqdm(STORIES_LIST, desc="Generating voices", ncols=80):
    try: 
        response = sarvamai_client.text_to_speech.convert(
            text=text,
            target_language_code="hi-IN",
            speaker="anushka",
            speech_sample_rate=16000,
            enable_preprocessing=True,
        )
    except Exception as e:
        print(f"Error generating speech for story {count}: {e}")
        continue
    save(response, f"audio_samples/{count}.wav")
    count += 1

Generating voices:   0%|                                | 0/100 [00:00<?, ?it/s]